# 06 · Power BI Export

This notebook creates the single order-level table used by the Power BI report. The dashboard stays downstream of the Python workflow and does not repeat the raw-data joins or model training.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('../data/processed/master_orders_clean_v2.csv')
PERSONA_PATH = Path('../outputs/interpretation/order_personas.csv')
PLACEMENT_PATH = Path('../outputs/predictions/placement_predictions.csv')
DELIVERY_PATH = Path('../outputs/predictions/delivery_predictions.csv')
POWERBI_DIR = Path('../data/powerbi')
POWERBI_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, low_memory=False)

In [ ]:
keep = [
    'order_id','order_purchase_timestamp','customer_state',
    'primary_product_category','n_items','total_price','total_freight',
    'total_payment_value','primary_payment_type','n_payment_installments',
    'promised_delivery_days','delivery_time_days',
    'delivery_vs_estimate_days','delivered_late','late_days',
    'review_bad','split'
]

bi = df[keep].copy()
bi['order_purchase_timestamp'] = pd.to_datetime(
    bi['order_purchase_timestamp'],
    errors='coerce'
)
bi['purchase_date'] = bi['order_purchase_timestamp'].dt.date.astype('string')
bi['purchase_year'] = bi['order_purchase_timestamp'].dt.year
bi['purchase_month'] = bi['order_purchase_timestamp'].dt.to_period('M').astype(str)

Only the columns needed by the dashboard are merged from downstream artifacts. The target stays as `review_bad`; duplicate copies of the actual outcome are not brought in from the prediction files.

In [ ]:
if PERSONA_PATH.exists():
    personas = pd.read_csv(PERSONA_PATH, usecols=['order_id','persona'])
    bi = bi.merge(personas, on='order_id', how='left')
else:
    bi['persona'] = 'Not exported yet'

if PLACEMENT_PATH.exists():
    placement = pd.read_csv(
        PLACEMENT_PATH,
        usecols=[
            'order_id','placement_probability',
            'placement_prediction','placement_threshold'
        ]
    )
    bi = bi.merge(placement, on='order_id', how='left')

if DELIVERY_PATH.exists():
    delivery = pd.read_csv(
        DELIVERY_PATH,
        usecols=[
            'order_id','delivery_probability',
            'delivery_prediction','delivery_threshold'
        ]
    )
    bi = bi.merge(delivery, on='order_id', how='left')

## Risk band and queue position

The risk band is tied to the model's selected operating threshold instead of fixed probability cutoffs. I also add a rank, percentile and top-risk flags for the scored test orders so Power BI can reproduce the capacity analysis without rebuilding the model logic.

In [ ]:
if {'delivery_probability','delivery_threshold'}.issubset(bi.columns):
    threshold = bi['delivery_threshold'].dropna().iloc[0]

    bi['risk_band'] = pd.cut(
        bi['delivery_probability'],
        bins=[-0.01, threshold / 2, threshold, 1.0],
        labels=['Low','Medium','High'],
        include_lowest=True
    )

    scored = bi['delivery_probability'].notna()
    bi.loc[scored, 'delivery_risk_rank'] = (
        bi.loc[scored, 'delivery_probability']
        .rank(method='first', ascending=False)
    )
    bi.loc[scored, 'delivery_risk_percentile'] = (
        bi.loc[scored, 'delivery_probability']
        .rank(method='average', pct=True)
    )

    bi['top_5pct_risk'] = False
    bi['top_10pct_risk'] = False
    bi['top_20pct_risk'] = False
    bi.loc[scored, 'top_5pct_risk'] = bi.loc[scored, 'delivery_risk_percentile'] >= 0.95
    bi.loc[scored, 'top_10pct_risk'] = bi.loc[scored, 'delivery_risk_percentile'] >= 0.90
    bi.loc[scored, 'top_20pct_risk'] = bi.loc[scored, 'delivery_risk_percentile'] >= 0.80

    bi['service_priority'] = 'Not scored'
    bi.loc[scored, 'service_priority'] = 'Outside top 20%'
    bi.loc[scored & bi['top_20pct_risk'], 'service_priority'] = 'Top 20%'
    bi.loc[scored & bi['top_10pct_risk'], 'service_priority'] = 'Top 10%'
    bi.loc[scored & bi['top_5pct_risk'], 'service_priority'] = 'Top 5%'

In [ ]:
OUT = POWERBI_DIR / 'powerbi_order_risk.csv'
bi.to_csv(OUT, index=False)

print('Saved:', OUT)
print('Rows:', len(bi), 'Columns:', len(bi.columns))
bi.head()